# 📓 Ensemble Models: Decision Trees, Random Forests, & XGBoost

This notebook is the hands-on companion to **Part 3** of our Ensemble Models series. We will build a complete machine learning pipeline to train, tune, and evaluate:
1. **Decision Trees** (Baseline model)
2. **Random Forests** (Bagging ensemble)
3. **XGBoost** (Gradient Boosting ensemble)

We will use a synthetic customer churn dataset to demonstrate data preprocessing, hyperparameter search using cross-validation, and model interpretation.

---  
## 🛠️ 1. Setup and Library Imports

First, we import the standard data science and machine learning libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, 
    accuracy_score, 
    roc_auc_score, 
    roc_curve, 
    confusion_matrix
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# Formatting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
np.random.seed(42)

---  
## 📊 2. Synthetic Dataset Generation

To simulate a real-world customer churn scenario, we will generate a dataset containing both continuous and categorical features. We will also inject some missing values (`NaN`) to show how different models handle them.

In [ ]:
# 1. Create a classification dataset with 20 features
X_raw, y_raw = make_classification(
    n_samples=2000, 
    n_features=12, 
    n_informative=8, 
    n_redundant=4, 
    random_state=42
)

# 2. Convert to a DataFrame and add descriptive column names
feature_names = [
    "age", "tenure", "monthly_charges", "total_charges",
    "support_tickets", "login_frequency", "usage_metric_a", "usage_metric_b",
    "usage_metric_c", "usage_metric_d", "usage_metric_e", "usage_metric_f"
]
df = pd.DataFrame(X_raw, columns=feature_names)
df["churn"] = y_raw

# 3. Convert some numeric columns to categories
# E.g., splitting a feature into 'contract_type' and 'payment_method'
df["contract_type"] = pd.qcut(df["usage_metric_e"], q=3, labels=["Month-to-month", "One year", "Two year"])
df["payment_method"] = pd.qcut(df["usage_metric_f"], q=3, labels=["Electronic check", "Credit card", "Bank transfer"])

# Drop the original columns used for binning
df = df.drop(columns=["usage_metric_e", "usage_metric_f"])

# 4. Inject missing values (NaN) into 'monthly_charges' and 'tenure'
# E.g., simulating incomplete records
mask_charges = np.random.rand(*df["monthly_charges"].shape) < 0.08
df.loc[mask_charges, "monthly_charges"] = np.nan

mask_tenure = np.random.rand(*df["tenure"].shape) < 0.05
df.loc[mask_tenure, "tenure"] = np.nan

df.head()

---  
## 🧪 3. Data Splitting & Pipeline Preprocessing

We separate our target column `churn` and split our dataset into **80% Training** and **20% Testing**.

Since tree models are scale-invariant, we **do not need to scale our features**. We only need to:
1. **Impute missing values** (for Scikit-Learn trees).
2. **Encode categorical values** into numbers.

In [ ]:
# Split target and features
X = df.drop(columns=["churn"])
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

### Creating the Preprocessing Pipeline

In [ ]:
# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

# 1. Numeric pipeline: Impute missing values with the median
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# 2. Categorical pipeline: Ordinal Encode categorical columns (assigning integer labels)
categorical_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Preprocess the datasets
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Get updated feature order for interpreting feature importances later
all_features = numeric_features + categorical_features

---  
## 🌳 4. Baseline: Decision Tree

We train a simple decision tree with a `max_depth` limit of 3 to visualize how the splitting rules are constructed.

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train_preprocessed, y_train)

# Plot the tree structure
plt.figure(figsize=(18, 10))
plot_tree(
    dt_model, 
    feature_names=all_features, 
    class_names=["Stay", "Churn"], 
    filled=True, 
    rounded=True, 
    fontsize=12
)
plt.title("Visualizing Decision Tree Splits (Depth=3)", fontsize=16)
plt.show()

---  
## 🌲 5. Parallel Ensembles: Random Forest & Hyperparameter Tuning

We will tune a **Random Forest Classifier** using **RandomizedSearchCV** with a 5-Fold cross-validation strategy. We will also access the built-in **Out-of-Bag (OOB) Score**.

In [ ]:
# Define the parameter grid to sample from
rf_param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [5, 10, 15, 20, None],
    'max_features': ['sqrt', 'log2', 0.5, 0.8],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize base model with bootstrap enabled to compute OOB score
rf_base = RandomForestClassifier(bootstrap=True, oob_score=True, random_state=42)

# Setup Randomized Search
rf_search = RandomizedSearchCV(
    estimator=rf_base, 
    param_distributions=rf_param_grid,
    n_iter=20, 
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_preprocessed, y_train)

print(f"Best Random Forest Parameters: {rf_search.best_params_}")
print(f"Best Cross-Validation ROC-AUC: {rf_search.best_score_:.4f}")

### OOB (Out-of-Bag) Score Verification
Let's extract the best estimator and check its OOB score. Since OOB acts as a built-in validation set, it should be very close to the cross-validation score.

In [ ]:
best_rf = rf_search.best_estimator_
print(f"Random Forest OOB Accuracy Score: {best_rf.oob_score_:.4f}")

---  
## ⚡ 6. Sequential Ensembles: XGBoost & Native Missing Data

XGBoost handles missing values (`NaN`) natively. To demonstrate this, we will train XGBoost using **raw data** (where categorical strings are Ordinal Encoded, but numeric missing values are **not imputed**).

In [ ]:
# Preprocess data WITHOUT numeric imputation
xgb_preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', numeric_features), # Leave NaNs intact
    ('cat', categorical_transformer, categorical_features)
])

X_train_xgb = xgb_preprocessor.fit_transform(X_train)
X_test_xgb = xgb_preprocessor.transform(X_test)

### Tuning XGBoost
We define the tuning space for XGBoost. We tune `learning_rate`, `max_depth`, row/column subsampling, and regularization weights.

In [ ]:
xgb_param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_lambda': [0.1, 1.0, 10.0],
    'reg_alpha': [0.0, 0.1, 1.0],
    'n_estimators': [100, 200, 300]
}

xgb_base = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss', 
    random_state=42
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=xgb_param_grid,
    n_iter=20, 
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)

xgb_search.fit(X_train_xgb, y_train)

print(f"Best XGBoost Parameters: {xgb_search.best_params_}")
print(f"Best Cross-Validation ROC-AUC: {xgb_search.best_score_:.4f}")

---  
## 📊 7. Model Comparison on Test Data

Let's compare our models on the held-out test dataset.

In [ ]:
# Get predictions
dt_preds = dt_model.predict(X_test_preprocessed)
rf_preds = best_rf.predict(X_test_preprocessed)
xgb_preds = xgb_search.best_estimator_.predict(X_test_xgb)

print("=== Decision Tree Classification Report ===")
print(classification_report(y_test, dt_preds))

print("=== Random Forest Classification Report ===")
print(classification_report(y_test, rf_preds))

print("=== XGBoost Classification Report ===")
print(classification_report(y_test, xgb_preds))

### Plotting ROC Curves

In [ ]:
# Get probability scores
dt_probs = dt_model.predict_proba(X_test_preprocessed)[:, 1]
rf_probs = best_rf.predict_proba(X_test_preprocessed)[:, 1]
xgb_probs = xgb_search.best_estimator_.predict_proba(X_test_xgb)[:, 1]

# Compute ROC statistics
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_probs)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_probs)
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_probs)

# Plot
plt.figure(figsize=(10, 7))
plt.plot(dt_fpr, dt_tpr, label=f"Decision Tree (AUC = {roc_auc_score(y_test, dt_probs):.4f})")
plt.plot(rf_fpr, rf_tpr, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_probs):.4f})")
plt.plot(xgb_fpr, xgb_tpr, label=f"XGBoost (AUC = {roc_auc_score(y_test, xgb_probs):.4f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison", fontsize=16)
plt.legend(loc="lower right")
plt.show()

---  
## 🔍 8. Model Interpretation: Feature Importances

Let's check and compare the feature importance calculations from both Random Forest and XGBoost.

In [ ]:
# Extract importances
rf_importances = best_rf.feature_importances_
xgb_importances = xgb_search.best_estimator_.feature_importances_

# Combine into a DataFrame
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Random Forest': rf_importances,
    'XGBoost': xgb_importances
}).melt(id_vars='Feature', var_name='Model', value_name='Importance')

# Plot comparison
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df, x='Importance', y='Feature', hue='Model')
plt.title("Feature Importance Comparison: RF vs. XGBoost", fontsize=16)
plt.xlabel("Relative Importance")
plt.show()

---  
## 🏁 9. Key Takeaways

1. **Decision Tree (Baseline)**: Simple and highly interpretable, but lacks performance on complex datasets.
2. **Random Forest (Bagging)**: Combats variance by training independent deep trees. Very robust and easy to tune, but struggles slightly with native missing values.
3. **XGBoost (Boosting)**: Sequential model optimizing residuals. Achieved the highest performance (AUC), handles missing values natively, and runs extremely fast due to parallel split optimizations. However, it requires careful hyperparameter tuning to prevent overfitting.